# Limpieza y Filtrado del Dataset de Películas

En el notebook anterior (EDA) se analizó la estructura y calidad del dataset de películas. En este notebook realizaremos la **limpieza** y el **filtrado** para quedarnos únicamente con películas relevantes para el sistema de recomendación.

El objetivo es obtener un catálogo de **5,000 películas de calidad** que sirva como base para el recomendador basado en contenido.

In [1]:
import pandas as pd
import ast
import numpy as np
import warnings
warnings.filterwarnings("ignore")

## Dataset de Películas

In [2]:
mdf = pd.read_parquet('../data/raw/movies_metadata.parquet')
mdf.columns

Index(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres',
       'homepage', 'id', 'imdb_id', 'origin_country', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title',
       'video', 'vote_average', 'vote_count', 'movieId'],
      dtype='object')

En este dataset `movieId` corresponde al identificador que le asignó MovieLens y `id` corresponde al identificador que le asignó **TMDB**

In [3]:
# Select only the relevant columns
mdf = mdf[['movieId', 'id', 'title', 'genres', 'overview', 
           'release_date', 'runtime', 'tagline',  'vote_average', 
           'popularity', 'vote_count', 'poster_path', 'backdrop_path']]

mdf.head().transpose()

,0,1,2,3,4
movieId,10,8,6,7,1
id,710,45325,949,11860,862
title,GoldenEye,Tom and Huck,Heat,Sabrina,Toy Story
genres,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...","[{'id': 10751, 'name': 'Family'}, {'id': 28, '...","[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...","[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...","[{'id': 10751, 'name': 'Family'}, {'id': 35, '..."
overview,When a powerful satellite system falls into th...,"A mischievous young boy, Tom Sawyer, witnesses...",Obsessive master thief Neil McCauley leads a t...,"After her return from school in Paris, a playb...","Led by Woody, Andy's toys live happily in his ..."
release_date,1995-11-16,1995-12-22,1995-12-15,1995-12-15,1995-11-22
runtime,130,97,170,127,81
tagline,No limits. No fears. No substitutes.,A lot of kids get into trouble. These two inve...,A Los Angeles crime saga.,You are cordially invited to the most surprisi...,The adventure takes off when toys come to life!
vote_average,6.9,5.288,7.931,6.214,7.971
popularity,9.0518,1.1156,15.136,3.876,20.4112


In [4]:
mdf.dtypes

movieId            int64
id                 int64
title             object
genres            object
overview          object
release_date      object
runtime            int64
tagline           object
vote_average     float64
popularity       float64
vote_count         int64
poster_path       object
backdrop_path     object
dtype: object

Los tipos de datos de todas las columnas son correctos.

In [5]:
print(f'The original movies dataset has {mdf.shape[0]:,} movies')

The original movies dataset has 86,242 movies


## Limpieza y Preprocesamiento

Eliminamos filas que carecen de información esencial para el sistema de recomendación:

- **`title`**: sin título no podemos identificar la película
- **`overview`**: necesario para el recomendador basado en contenido (embeddings de descripción)
- **`genres`**: sin géneros no podemos calcular perfiles de personalidad de los usuarios

Estas columnas son obligatorias para el pipeline completo.

In [6]:
# Drop films without title or genre
mdf.dropna(subset='title', inplace=True)
mdf.dropna(subset='overview', inplace=True)

In [7]:
# Extract the genres
mdf['genres'] =  mdf['genres'].apply(lambda x: [item['name'] for item in x])
mdf.head().transpose()

,0,1,2,3,4
movieId,10,8,6,7,1
id,710,45325,949,11860,862
title,GoldenEye,Tom and Huck,Heat,Sabrina,Toy Story
genres,"[Adventure, Action, Thriller]","[Family, Action, Adventure, Drama]","[Crime, Drama, Action]","[Romance, Drama, Comedy]","[Family, Comedy, Animation, Adventure]"
overview,When a powerful satellite system falls into th...,"A mischievous young boy, Tom Sawyer, witnesses...",Obsessive master thief Neil McCauley leads a t...,"After her return from school in Paris, a playb...","Led by Woody, Andy's toys live happily in his ..."
release_date,1995-11-16,1995-12-22,1995-12-15,1995-12-15,1995-11-22
runtime,130,97,170,127,81
tagline,No limits. No fears. No substitutes.,A lot of kids get into trouble. These two inve...,A Los Angeles crime saga.,You are cordially invited to the most surprisi...,The adventure takes off when toys come to life!
vote_average,6.9,5.288,7.931,6.214,7.971
popularity,9.0518,1.1156,15.136,3.876,20.4112


Eliminamos las películas sin géneros, ya que son indispensables para el sistema de personalidad.

In [8]:
mdf['genres'] = mdf['genres'].apply(lambda x: np.nan if not x else x)
mdf.dropna(subset='genres', inplace=True)

#### Poster y Backdrop

El poster y el backdrop son necesarios para mostrar las películas en la interfaz de usuario. Verificamos cuántas películas carecen de estas imágenes.

In [9]:
mdf['poster_path'].isnull().sum()

np.int64(920)

In [10]:
mdf['backdrop_path'].isnull().sum()

np.int64(9603)

In [11]:
# Sort by popularity the movies that do not have a poster path
mdf[mdf['poster_path'].isnull()].sort_values(by='popularity', ascending=False)[['title', 'popularity']].head(10)

,title,popularity
47216,The Void,2.6568
39589,Gestalt,1.3407
58776,Facade,1.0769
69570,The Water Engine,1.0518
50243,The Fool,1.0480
76050,Forfeit,0.9967
45980,The Kopeck,0.9700
27717,Secrets of a Married Man,0.9682
60605,An Assassin,0.9351
60358,The Zoo Gang,0.8979


In [12]:
mdf[mdf['backdrop_path'].isnull()].\
    sort_values(by='popularity', ascending=False)[['title', 'popularity', 'vote_average', 'vote_count']].head(10)

,title,popularity,vote_average,vote_count
24199,No Thank You,4.8135,4.5,15
28029,Terror on the 40th Floor,3.1377,4.2,8
72106,Bomb Girls: Facing the Enemy,3.0926,6.3,14
29661,Reflections of Light,2.9318,6.0,13
47216,The Void,2.6568,4.2,5
36559,Mr. Kuka's Advice,2.5180,5.9,6
71664,Fig,2.4459,6.0,5
85342,Rehearsal for Murder,2.3904,6.1,17
58551,Dead Mine,2.3214,4.8,125
31789,Filumena Marturano,2.1213,8.8,6


Las películas sin poster tienden a ser poco populares y con muy pocos votos. Dado que el poster es imprescindible para la interfaz de usuario, eliminamos las películas que no cuenten con él. Lo mismo aplica para el backdrop.

In [13]:
mdf.dropna(subset='poster_path', inplace=True)

In [14]:
mdf.dropna(subset='backdrop_path', inplace=True)

In [15]:
print(f'After doing some cleaning we are left with {mdf.shape[0]:,} movies')

After doing some cleaning we are left with 75,837 movies


## Filtrado del Catálogo

Para mejorar la calidad del sistema de recomendación, seleccionamos únicamente películas relevantes. Filtrar el catálogo tiene varias ventajas:

- **Relevancia para el público actual:** Las películas más recientes y populares se alinean mejor con las preferencias de los usuarios jóvenes.
- **Evitar la dispersión de datos (*data sparsity*):** Un catálogo más pequeño y de calidad genera interacciones más densas, lo que mejora el rendimiento de los modelos colaborativos.
- **Mejor calidad de contenido:** Las películas con más votos tienen información más confiable para los embeddings del recomendador basado en contenido.

Filtrar películas únicamente por número de votos, popularidad o calificación promedio puede llevar a resultados engañosos. Para resolver esto, aplicamos la **fórmula de calificación ponderada de IMDb**, que balancea el promedio de calificaciones con la cantidad de votos. Una calificación alta por sí sola no es suficiente: la película también necesita un número significativo de votos para ser confiable.

$$\text{WR} = \left( \frac{v}{v+m} \right) R + \left( \frac{m}{v+m} \right) C$$

Donde:
- $v$ = número de votos de la película
- $m$ = mínimo de votos requerido (percentil 90)
- $R$ = calificación promedio de la película
- $C$ = calificación promedio de todas las películas

In [16]:
m = mdf['vote_count'].quantile(0.9)
C = mdf['vote_average'].mean()

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v / (v + m) * R) + (m / (v + m) * C)

In [17]:
mdf['score'] = mdf.apply(weighted_rating, axis=1)

# Top 5 movies according to IMDb's score
mdf.sort_values(by='score', ascending=False).head()

,movieId,id,title,genres,overview,release_date,runtime,tagline,vote_average,popularity,vote_count,poster_path,backdrop_path,score
313,318,278,The Shawshank Redemption,"[Drama, Crime]",Imprisoned in the 1940s for the double murder ...,1994-09-23,142,Fear can hold you prisoner. Hope can set you f...,8.718,41.1463,30088,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,8.675355
833,858,238,The Godfather,"[Drama, Crime]","Spanning the years 1945 to 1955, a chronicle o...",1972-03-14,175,An offer you can't refuse.,8.687,39.2262,22736,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,/tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg,8.631468
521,527,424,Schindler's List,"[Drama, History, War]",The true story of how businessman Oskar Schind...,1993-12-15,195,"Whoever saves one life, saves the world entire.",8.567,21.5504,17300,/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,8.497640
12165,58559,155,The Dark Knight,"[Action, Crime, Thriller]",Batman raises the stakes in his war on crime. ...,2008-07-16,152,Welcome to a world without rules.,8.528,34.1491,35465,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,/cfT29Im5VDvjE0RpyKOSdCKZal7.jpg,8.494211
1185,1221,240,The Godfather Part II,"[Drama, Crime]",In the continuing saga of the Corleone crime f...,1974-12-20,202,The rise and fall of the Corleone empire.,8.572,22.3481,13775,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,8.485309


El filtrado por score ponderado produce resultados de alta calidad: las películas con mayor score son títulos ampliamente reconocidos y valorados por el público.

### Filtro por Año y Calificación

Seleccionamos películas estrenadas **a partir de 1995**, ya que:
- La mayoría de nuestros usuarios objetivo son jóvenes nacidos en los 2000s
- Las películas anteriores a 1995 tienen muy pocos ratings en el dataset de MovieLens
- La calidad de la información en TMDB es significativamente mejor para películas modernas

In [18]:
mdf['release_date'] = pd.to_datetime(mdf['release_date'], errors='coerce')
mdf['year'] = mdf['release_date'].dt.year.fillna(1989).astype('int')
# Drop the release date
mdf = mdf.drop(columns=['release_date']).reset_index(drop=True)
# Filter the movies
mdf = mdf[mdf['year'] > 1994]

Seleccionamos las mejores **5,000 películas** según el score ponderado. Antes de aplicar el `.head(5000)` realizamos la eliminación de duplicación por título (conservando la versión con mayor score) para garantizar exactamente 5,000 películas distintas.

In [19]:
# Deduplicate by title (keep highest score) before selecting top 5,000
# This prevents losing movies later in Notebook 4's dedup step
mdf = mdf.loc[mdf.groupby('title')['score'].idxmax()]

mdf = mdf.sort_values(by='score', ascending=False).head(5000)

# Ensure the size is correct
mdf.shape

(5000, 14)

### Recálculo del Score sobre el Catálogo Filtrado

El `score` calculado anteriormente usaba los parámetros `m` y `C` derivados del dataset **completo de ~87,000 películas**. Ahora que hemos filtrado el catálogo, la distribución de votos cambió significativamente (la mediana pasó de ~10 a ~960 votos), por lo que recalculamos el score con los nuevos parámetros para que sea más preciso y discriminativo dentro de este conjunto de alta calidad.

In [20]:
m = mdf['vote_count'].quantile(0.9)
C = mdf['vote_average'].mean()

mdf['score'] = mdf.apply(weighted_rating, axis=1, m=m, C=C)

print(f'Recalculated on filtered catalog: m={m:.0f} votes, C={C:.3f}')
mdf.sort_values(by='score', ascending=False).head(10)[['title', 'score', 'vote_count', 'vote_average']]

Recalculated on filtered catalog: m=7295 votes, C=7.179


,title,score,vote_count,vote_average
11813,The Dark Knight,8.297891,35465,8.528
19967,Interstellar,8.268127,39350,8.470
6766,The Lord of the Rings: The Return of the King,8.210106,26305,8.496
14361,Inception,8.207556,38982,8.400
2734,Fight Club,8.171999,31765,8.400
4701,The Lord of the Rings: The Fellowship of the Ring,8.167058,27303,8.431
53915,Parasite,8.147664,20399,8.494
5298,Spirited Away,8.144856,18149,8.533
2921,The Green Mile,8.137981,19057,8.505
5617,The Lord of the Rings: The Two Towers,8.123974,23682,8.415


In [ ]:
mdf.head(10)

,movieId,id,title,genres,overview,runtime,tagline,vote_average,popularity,vote_count,poster_path,backdrop_path,score,year
11813,58559,155,The Dark Knight,"[Action, Crime, Thriller]",Batman raises the stakes in his war on crime. ...,152,Welcome to a world without rules.,8.528,34.1491,35465,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,/cfT29Im5VDvjE0RpyKOSdCKZal7.jpg,8.297891,2008
5298,5618,129,Spirited Away,"[Animation, Family, Fantasy]","A young girl, Chihiro, becomes trapped in a st...",125,Beyond the tunnel was a mysterious town.,8.533,30.1776,18149,/39wmItIWsg5sZMyRUHLkWBcuVCM.jpg,/dyJvKsNs2KP8qQnAXbRwDjblViy.jpg,8.144856,2001
6766,7153,122,The Lord of the Rings: The Return of the King,"[Adventure, Fantasy, Action]",As armies mass for a final battle that will de...,201,The eye of the enemy is moving.,8.496,25.4461,26305,/rCzpDGLbOoPwLjy3OAm5NUPOTrC.jpg,/2u7zbn8EudG6kLlBzUYqP8RyFU4.jpg,8.210106,2003
2921,3147,497,The Green Mile,"[Fantasy, Drama, Crime]",A supernatural tale set on death row in a Sout...,189,Paul Edgecomb didn't believe in miracles. Unti...,8.505,22.4205,19057,/8VG8fDNiy50H4FedGwdSVUPoaJe.jpg,/b6HWTOxn1xevvyHU2K9ICvaRU6g.jpg,8.137981,1999
19967,109487,157336,Interstellar,"[Adventure, Drama, Science Fiction]",The adventures of a group of explorers who mak...,169,Mankind was born on Earth. It was never meant ...,8.470,65.7255,39350,/yQvGrMoipbRoddT0ZR8tPoR7NfX.jpg,/2ssWTSVklAEc98frZUQhgtGHx7s.jpg,8.268127,2014
53915,202439,496243,Parasite,"[Comedy, Thriller, Drama]","All unemployed, Ki-taek's family takes peculia...",133,Act like you own the place.,8.494,32.6696,20399,/7IiTTgloJzvGI1TAYymCfbfl3vT.jpg,/hiKmpZMGZsrkA3cdce8a7Dpos1j.jpg,8.147664,2019
4701,4993,120,The Lord of the Rings: The Fellowship of the Ring,"[Adventure, Fantasy, Action]","Young hobbit Frodo Baggins, after inheriting a...",179,One ring to rule them all.,8.431,29.9709,27303,/6oom5QYQ2yQTMJIbnvbkBL9cHo6.jpg,/oiwc338EoBgS4sEI2ixAny4KQKg.jpg,8.167058,2001
38353,163134,372058,Your Name.,"[Animation, Romance, Drama]",High schoolers Mitsuha and Taki are complete s...,106,"Separated by distance, connected by fate.",8.478,24.7768,12414,/q719jXXEzOoYaps6babgKnONONX.jpg,/8x9iKH8kWA0zdkgNdpAew7OstYe.jpg,7.997270,2016
14361,79132,27205,Inception,"[Action, Science Fiction, Adventure]","Cobb, a skilled thief who commits corporate es...",148,Your mind is the scene of the crime.,8.400,28.6011,38982,/xlaY2zyzMfkhk0HSC5VUwzoZPU1.jpg,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,8.207556,2010
5617,5952,121,The Lord of the Rings: The Two Towers,"[Adventure, Fantasy, Action]",Frodo Baggins and the other members of the Fel...,179,The journey continues.,8.415,20.3212,23682,/5VTN0pR8gcqV3EPUHHfMGnJYN9L.jpg,/kWYfW2Re0rUDE6IHhy4CRuKWeFr.jpg,8.123974,2002


El catálogo final de 5,000 películas contiene títulos reconocidos y bien valorados, lo que debería mejorar significativamente la calidad de las recomendaciones. Este tamaño de catálogo ofrece un buen balance entre cobertura y densidad de interacciones para los modelos de filtrado colaborativo.

Dado que ya tenemos la información de las películas en `raw/movies_metadata.parquet` solo vamos a guardar el `id` y `movieId` de las películas en un archivo llamado `processed/clean_movies_ids.csv`

In [22]:
# Save only the ids on a csv file since we already have its metadata in another file
mdf[['movieId', 'id']].rename(columns={'movieId': 'movielens_id', 'id': 'tmdb_id'}).to_csv(
    '../data/processed/clean_movies_ids.csv', index=False
)

## Resumen del Notebook

En este notebook se limpió y filtró el dataset de películas para construir un catálogo de calidad.

**Acciones realizadas:**
- Eliminación de películas sin título, sinopsis, géneros, poster o backdrop
- Cálculo del score de calidad con la fórmula de calificación ponderada de IMDb
- Filtro por año (≥ 1995)
- Deduplicación por título (conservando el de mayor score)
- Recálculo del score sobre el catálogo filtrado

**Resultado:** Catálogo final de **5,000 películas**

**Output generado:** `ml/data/processed/clean_movies_ids.csv`